In [1]:
import os
import requests
import json
from pathlib import Path
from dotenv import load_dotenv

print("Python environment ready")

Python environment ready


In [2]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\venv\Scripts\python.exe


In [3]:
import requests

print("requests:", requests.__version__)
print("ALL CORE PACKAGES OK")

requests: 2.32.3
ALL CORE PACKAGES OK


In [4]:
load_dotenv()

API_KEY = os.getenv("XENO_CANTO_API_KEY")

if API_KEY:
    print("Xeno-canto API key loaded successfully.")
else:
    print("ERROR: XENO_CANTO_API_KEY was not found.")

Xeno-canto API key loaded successfully.


In [5]:
BASE_URL = "https://xeno-canto.org/api/3/recordings"

print(BASE_URL)

https://xeno-canto.org/api/3/recordings


In [6]:
params = {
    "query": 'grp:birds box:32.5,-124.5,42.0,-114.1 q:">C"',
    "key": API_KEY,
    "page": 1,
    "per_page": 5,
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("content-type"))

Status code: 200
Content type: application/json


In [7]:
response.raise_for_status()

data = response.json()

print("Response type:", type(data))
print("\nTop-level keys:")
print(list(data.keys()))

Response type: <class 'dict'>

Top-level keys:
['numRecordings', 'numSpecies', 'page', 'numPages', 'recordings']


In [8]:
recordings = data["recordings"]

print("Number of recordings returned:", len(recordings))

if recordings:
    print("\nFirst recording fields:")
    print(list(recordings[0].keys()))

Number of recordings returned: 50

First recording fields:
['id', 'gen', 'sp', 'ssp', 'grp', 'status', 'en', 'rec', 'cnt', 'loc', 'lat', 'lon', 'alt', 'type', 'sex', 'stage', 'method', 'url', 'file', 'file-name', 'sono', 'osci', 'lic', 'q', 'length', 'time', 'date', 'uploaded', 'also', 'rmk', 'animal-seen', 'playback-used', 'temp', 'regnr', 'auto', 'dvc', 'mic', 'smp', 'annotation-set']


In [9]:
recording = recordings[0]

fields_to_check = [
    "id",
    "gen",
    "sp",
    "ssp",
    "grp",
    "status",
    "en",
    "rec",
    "cnt",
    "loc",
    "lat",
    "lon",
    "alt",
    "type",
    "method",
    "file",
    "file-name",
    "sono",
    "osci",
    "lic",
    "q",
    "length",
    "time",
    "date",
    "uploaded",
    "rmk",
    "animal-seen",
    "playback-used",
    "temp",
    "auto",
    "dvc",
    "mic",
    "smp",
]

for field in fields_to_check:
    print(f"{field:18} : {recording.get(field)}")

id                 : 384977
gen                : Larus
sp                 : brachyrhynchus
ssp                : 
grp                : birds
status             : identified
en                 : Short-billed Gull
rec                : Paul Marvin
cnt                : United States
loc                : Ferndale Bottoms, Ferndale, Humboldt County, California
lat                : 40.5942
lon                : -124.2411
alt                : 10
type               : call, flight call
method             : field recording
file               : https://xeno-canto.org/384977/download
file-name          : XC384977-Mew Gull adult, flt end -CA, Ferndale Bottoms, Humboldt County, January 27, 2017, 0446 PM.mp3
sono               : {'small': 'https://xeno-canto.org/sounds/spectrograms/RFTXRYBVBX/384977/grey-small.png', 'med': 'https://xeno-canto.org/sounds/spectrograms/RFTXRYBVBX/384977/grey-medium.png', 'large': 'https://xeno-canto.org/sounds/spectrograms/RFTXRYBVBX/384977/colour.png', 'full': 'https://xe

In [10]:
print("API reports:")
print("Total recordings:", data["numRecordings"])
print("Total species:", data["numSpecies"])
print("Current page:", data["page"])
print("Number of pages:", data["numPages"])

print("\nActual recordings in this response:", len(data["recordings"]))

API reports:
Total recordings: 10923
Total species: 411
Current page: 1
Number of pages: 219

Actual recordings in this response: 50


In [11]:
print("Requested per_page:", params["per_page"])
print("Returned:", len(data["recordings"]))
print("Reported pages:", data["numPages"])

Requested per_page: 5
Returned: 50
Reported pages: 219


## Acquisition Strategy

The initial dataset focuses on bird vocalizations from California, Arizona,
and Texas. Xeno-canto recording metadata are first retrieved and preserved
before a smaller analytical subset is selected.

The acquisition process separates metadata retrieval from final recording
selection. This allows the dataset to be inspected for species, geographic,
temporal, recording-quality, and audio characteristics before downloading
the final subset of recordings.


In [12]:
STATE_BOXES = {
    "CA": "32.5,-124.5,42.0,-114.1",
    "AZ": "31.3,-114.9,37.0,-109.0",
    "TX": "25.8,-106.7,36.5,-93.5",
}

A1_TARGETS = {
    "CA": 100,
    "AZ": 75,
    "TX": 100,
}

print("Study regions:", list(STATE_BOXES.keys()))
print("Initial target:", sum(A1_TARGETS.values()), "recordings")

Study regions: ['CA', 'AZ', 'TX']
Initial target: 275 recordings


In [13]:
PROJECT_ROOT = Path.cwd().parents[2]

RAW_XC_DIR = PROJECT_ROOT / "data" / "raw" / "xenocanto"
RAW_AUDIO_DIR = PROJECT_ROOT / "audio" / "raw"

RAW_XC_DIR.mkdir(parents=True, exist_ok=True)
RAW_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw metadata directory:")
print(RAW_XC_DIR)

print("\nRaw audio directory:")
print(RAW_AUDIO_DIR)

Project root:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity

Raw metadata directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\xenocanto

Raw audio directory:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\audio\raw


## Raw Xeno-canto Metadata

The raw API response is preserved before any cleaning, filtering, transformation,
or feature engineering. Maintaining an immutable raw-data layer allows the
acquisition and preprocessing steps to remain reproducible and auditable.

In [14]:
import json
from datetime import datetime, timezone

raw_metadata_path = RAW_XC_DIR / "xc_api_validation_CA_page1.json"

raw_metadata = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "endpoint": BASE_URL,
    "query_parameters": {
        "query": params["query"],
        "page": params["page"],
        "per_page": params["per_page"],
    },
    "response": data,
}

with open(raw_metadata_path, "w", encoding="utf-8") as f:
    json.dump(raw_metadata, f, indent=2)

print(f"Saved raw API response to:")
print(raw_metadata_path)

Saved raw API response to:
c:\Users\Samanwita\Downloads\from-soundscapes-to-biodiversity\data\raw\xenocanto\xc_api_validation_CA_page1.json


In [15]:
print(raw_metadata_path.exists())
print(f"{raw_metadata_path.stat().st_size:,} bytes")

True
91,198 bytes


## Metadata Standardization

The raw Xeno-canto metadata contain abbreviated field names and heterogeneous
recording information. The metadata are standardized into descriptive,
analysis-ready variables while preserving the original raw response separately.

In [17]:
import pandas as pd
metadata_df = pd.DataFrame(recordings)

clean_metadata = pd.DataFrame({
    "recording_id": metadata_df["id"],
    "scientific_name": (
        metadata_df["gen"].fillna("") + " " +
        metadata_df["sp"].fillna("")
    ).str.strip(),
    "common_name": metadata_df["en"],
    "genus": metadata_df["gen"],
    "species": metadata_df["sp"],
    "subspecies": metadata_df["ssp"],
    "recordist": metadata_df["rec"],
    "country": metadata_df["cnt"],
    "location": metadata_df["loc"],
    "latitude": pd.to_numeric(metadata_df["lat"], errors="coerce"),
    "longitude": pd.to_numeric(metadata_df["lon"], errors="coerce"),
    "altitude_m": pd.to_numeric(metadata_df["alt"], errors="coerce"),
    "recording_type": metadata_df["type"],
    "method": metadata_df["method"],
    "quality": metadata_df["q"],
    "duration": metadata_df["length"],
    "date": metadata_df["date"],
    "time": metadata_df["time"],
    "sample_rate": pd.to_numeric(metadata_df["smp"], errors="coerce"),
    "audio_url": metadata_df["file"],
    "source_url": metadata_df["url"],
    "license": metadata_df["lic"],
})

print("Rows:", len(clean_metadata))
print("Columns:", len(clean_metadata.columns))

display(clean_metadata.head())

Rows: 50
Columns: 22


,recording_id,scientific_name,common_name,genus,species,subspecies,recordist,country,location,latitude,...,recording_type,method,quality,duration,date,time,sample_rate,audio_url,source_url,license
0,384977,Larus brachyrhynchus,Short-billed Gull,Larus,brachyrhynchus,,Paul Marvin,United States,"Ferndale Bottoms, Ferndale, Humboldt County, C...",40.5942,...,"call, flight call",field recording,B,0:24,2017-01-27,16:46,48000,https://xeno-canto.org/384977/download,https://xeno-canto.org/384977,https://creativecommons.org/licenses/by-nc-sa/...
1,163335,Branta bernicla,Brant Goose,Branta,bernicla,,Paul Marvin,United States,"Fiesta Island, San Diego, California",32.7802,...,call,field recording,A,0:08,2012-10-25,07:15,48000,https://xeno-canto.org/163335/download,https://xeno-canto.org/163335,https://creativecommons.org/licenses/by-nc-sa/...
2,299241,Branta bernicla,Brant Goose,Branta,bernicla,,Paul Marvin,United States,"San Diego River mouth by Robb Field, San Dieg...",32.7566,...,call,field recording,B,2:23,2015-04-11,08:42,48000,https://xeno-canto.org/299241/download,https://xeno-canto.org/299241,https://creativecommons.org/licenses/by-nc-sa/...
3,163352,Branta bernicla,Brant Goose,Branta,bernicla,,Paul Marvin,United States,"San Diego River mouth by Robb Field, San Dieg...",32.7566,...,flight call,field recording,B,0:08,2013-04-15,11:12,48000,https://xeno-canto.org/163352/download,https://xeno-canto.org/163352,https://creativecommons.org/licenses/by-nc-sa/...
4,163349,Branta bernicla,Brant Goose,Branta,bernicla,,Paul Marvin,United States,"Fiesta Island, San Diego, California",32.7802,...,call,field recording,B,0:03,2012-02-16,13:22,48000,https://xeno-canto.org/163349/download,https://xeno-canto.org/163349,https://creativecommons.org/licenses/by-nc-sa/...


In [18]:
clean_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recording_id     50 non-null     object 
 1   scientific_name  50 non-null     object 
 2   common_name      50 non-null     object 
 3   genus            50 non-null     object 
 4   species          50 non-null     object 
 5   subspecies       50 non-null     object 
 6   recordist        50 non-null     object 
 7   country          50 non-null     object 
 8   location         50 non-null     object 
 9   latitude         50 non-null     float64
 10  longitude        50 non-null     float64
 11  altitude_m       50 non-null     int64  
 12  recording_type   50 non-null     object 
 13  method           50 non-null     object 
 14  quality          50 non-null     object 
 15  duration         50 non-null     object 
 16  date             50 non-null     object 
 17  time             5

In [19]:
clean_metadata.isna().sum()

recording_id       0
scientific_name    0
common_name        0
genus              0
species            0
subspecies         0
recordist          0
country            0
location           0
latitude           0
longitude          0
altitude_m         0
recording_type     0
method             0
quality            0
duration           0
date               0
time               0
sample_rate        0
audio_url          0
source_url         0
license            0
dtype: int64

In [20]:
# Convert duration from MM:SS to seconds
def duration_to_seconds(value):
    if pd.isna(value):
        return None

    parts = str(value).split(":")

    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)

    elif len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

    return None


clean_metadata["duration_seconds"] = (
    clean_metadata["duration"].apply(duration_to_seconds)
)

# Convert date
clean_metadata["date"] = pd.to_datetime(
    clean_metadata["date"],
    errors="coerce"
)

# Convert time into a standardized string for now
clean_metadata["time"] = clean_metadata["time"].astype("string")

print(clean_metadata[[
    "recording_id",
    "duration",
    "duration_seconds",
    "date",
    "time"
]].head())

  recording_id duration  duration_seconds       date   time
0       384977     0:24              24.0 2017-01-27  16:46
1       163335     0:08               8.0 2012-10-25  07:15
2       299241     2:23             143.0 2015-04-11  08:42
3       163352     0:08               8.0 2013-04-15  11:12
4       163349     0:03               3.0 2012-02-16  13:22


In [21]:
clean_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   recording_id      50 non-null     object        
 1   scientific_name   50 non-null     object        
 2   common_name       50 non-null     object        
 3   genus             50 non-null     object        
 4   species           50 non-null     object        
 5   subspecies        50 non-null     object        
 6   recordist         50 non-null     object        
 7   country           50 non-null     object        
 8   location          50 non-null     object        
 9   latitude          50 non-null     float64       
 10  longitude         50 non-null     float64       
 11  altitude_m        50 non-null     int64         
 12  recording_type    50 non-null     object        
 13  method            50 non-null     object        
 14  quality           50 non-nul

In [22]:
print("Quality distribution:")
print(clean_metadata["quality"].value_counts())

print("\nRecording types:")
print(clean_metadata["recording_type"].value_counts().head(10))

print("\nSample rates:")
print(clean_metadata["sample_rate"].value_counts())

Quality distribution:
quality
B    39
A    11
Name: count, dtype: int64

Recording types:
recording_type
call                            25
flight call                     19
call, flight call                4
call, flight call, wing whir     1
call, wingbeats                  1
Name: count, dtype: int64

Sample rates:
sample_rate
48000    48
44100     2
Name: count, dtype: int64


In [23]:
# Temporal features for exploratory analysis

clean_metadata["year"] = clean_metadata["date"].dt.year
clean_metadata["month"] = clean_metadata["date"].dt.month
clean_metadata["day_of_year"] = clean_metadata["date"].dt.dayofyear

# Convert recording time to hour
clean_metadata["hour"] = pd.to_numeric(
    clean_metadata["time"].str.split(":").str[0],
    errors="coerce"
)

# Define broad seasons
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    return None

clean_metadata["season"] = clean_metadata["month"].apply(get_season)

display(
    clean_metadata[
        [
            "recording_id",
            "date",
            "year",
            "month",
            "hour",
            "season",
            "duration_seconds",
        ]
    ].head()
)

,recording_id,date,year,month,hour,season,duration_seconds
0,384977,2017-01-27,2017,1,16,Winter,24.0
1,163335,2012-10-25,2012,10,7,Fall,8.0
2,299241,2015-04-11,2015,4,8,Spring,143.0
3,163352,2013-04-15,2013,4,11,Spring,8.0
4,163349,2012-02-16,2012,2,13,Winter,3.0


In [24]:
print("Years:")
print(clean_metadata["year"].value_counts().sort_index())

print("\nSeasons:")
print(clean_metadata["season"].value_counts())

print("\nRecording hours:")
print(clean_metadata["hour"].describe())

Years:
year
2009     4
2012     6
2013     1
2014     1
2015     2
2016     2
2017    10
2018    14
2021     1
2023     2
2024     3
2025     2
2026     2
Name: count, dtype: int64

Seasons:
season
Winter    28
Fall      12
Spring    10
Name: count, dtype: int64

Recording hours:
count    50.000000
mean     10.100000
std       3.032141
min       3.000000
25%       8.000000
50%       9.500000
75%      13.000000
max      16.000000
Name: hour, dtype: float64


## Full Metadata Acquisition

Metadata are collected separately from audio files so that the available
recording pool can be audited before a final analytical subset is selected.
The acquisition process records the state, API page, retrieval timestamp,
and raw response for reproducibility.

In [25]:
import time
from datetime import datetime, timezone

def fetch_all_metadata(state, max_pages=None):
    """
    Retrieve Xeno-canto metadata for one study region.

    Raw API responses are returned as a list of page-level dictionaries.
    Audio files are not downloaded at this stage.
    """
    
    query = f'grp:birds box:{STATE_BOXES[state]} q:">C"'
    
    all_pages = []
    
    page = 1
    
    while True:
        params = {
            "query": query,
            "key": API_KEY,
            "page": page,
        }
        
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=30
        )
        
        response.raise_for_status()
        
        page_data = response.json()
        
        all_pages.append({
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "state": state,
            "page": page,
            "response": page_data
        })
        
        print(
            f"{state}: page {page}/{page_data['numPages']} | "
            f"{len(page_data['recordings'])} recordings"
        )
        
        if page >= page_data["numPages"]:
            break
        
        if max_pages is not None and page >= max_pages:
            break
        
        page += 1
        time.sleep(1)
    
    return all_pages

In [26]:
test_pages = fetch_all_metadata("CA", max_pages=2)

CA: page 1/110 | 100 recordings
CA: page 2/110 | 100 recordings


In [27]:
print("Pages collected:", len(test_pages))
print("Total recordings collected:",
      sum(len(p["response"]["recordings"]) for p in test_pages))

Pages collected: 2
Total recordings collected: 200
